In [5]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage , BaseMessage
from langgraph.checkpoint.memory import MemorySaver




class ChatState(TypedDict):

    messages : Annotated[list[BaseMessage], add_messages]


llm = ChatGroq(
    model = "openai/gpt-oss-20b"
)

def chat_node(state:ChatState):
    #store
    messages = state['messages']

    #respond
    response = llm.invoke(messages)

    #return      
    return {'messages':[response]}

  
checkpointer = MemorySaver()
graph = StateGraph(ChatState)



graph.add_node('chat_node', chat_node )
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

chatbot

initial_state = {
    'messages' : [HumanMessage(content ='what is the capital of pakistan')]
}

thread_id = '1'

config = {'configurable': {'thread_id': thread_id}}

chatbot.invoke(initial_state, config=config)

thread_id = '1'

while True:

    user_messages = input('Type here: ')

    print('MASTER:' , user_messages)

    if user_messages.strip().lower() in ['end','bye','tata']:
     break 

    config = {'configurable':{'thread_id': thread_id}}

    response = chatbot.invoke({'messages':[HumanMessage(content=user_messages)]} , config = config)

    print ('AI: ', response['messages'][-1].content)

  



MASTER: tata


In [9]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='what is the capital of pakistan', additional_kwargs={}, response_metadata={}, id='2882b4e2-e012-46d5-b16f-dd235c84329b'), AIMessage(content='The capital of Pakistan is **Islamabad**.', additional_kwargs={'reasoning_content': 'User asks: "what is the capital of Pakistan". Straight answer: Islamabad.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 77, 'total_tokens': 113, 'completion_time': 0.04288878, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.00430629, 'prompt_tokens_details': None, 'queue_time': 0.303035882, 'total_time': 0.04719507}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0af22-3a13-74f1-8fe9-112de5f856fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 36, 'total_tokens': 11